# I) Reference simulation

## 1) Geometry

First, we define the geometry, material distribution, and generate the mesh.

In [ ]:
pole_pairs = 6
bundles_per_half_slot = 5

# Generate the corresponding mesh
from utils.geometry import machine_mesh
mesh = machine_mesh(p=pole_pairs, 
                    bundles_per_half_slot=bundles_per_half_slot, 
                    hBundle=0.25e-3,
                    )

print(f"Generated mesh with {mesh.nv} nodes, {mesh.ne} elements")

In [ ]:
# Display the mesh
from ngsolve.webgui import Draw
Draw(mesh.Curve(2), filename = 'scenes/ref/mesh.html')

In [ ]:
# Display the different material distribution
air = 0
magnet = 1
iron = 2
copper = 3

zones = mesh.MaterialCF({"rotor" : magnet,
                         "core_stator" : iron,
                         "slot.*_bundle.*" : copper,
                         }, default = air)

Draw(zones, mesh,
     settings = {"Objects" : {"Wireframe" : False}},
     filename = "scenes/ref/materials.html")

In [ ]:
winding_type = "distributed" # "distributed" or "concentrated"
Irms = 5

materials = mesh.MaterialCF({"rotor" : 1,  "slot1.*_bundle.*" : 4, "slot2.*_bundle.*" : 1.5, "slot3.*_bundle.*" : 3, "core_stator" : 5})
Draw(materials, mesh,
     settings = {"Objects" : {"Wireframe" : False}},
     filename = "materials.html")

_______
## 2) Physical state computation

### 2.a) Material properties

In [ ]:
# Magnetic permeability
mur_iron = 1000

from ngsolve import pi
mu0 = 4e-7 * pi

# Define reluctivity necessary for the magnetic vector-potential formulation
nu = mesh.MaterialCF({"core_stator" : 1 / (mu0 * mur_iron)}, default = 1/(mu0 * pi)) 

# Display relative permeability
print("Relative permeability")
Draw(1/(mu0*nu), mesh,
     settings = {"Objects" : {"Wireframe" : False}},
     filename = "scenes/ref/mu_r.html")

In [ ]:
# Electrical conductivity

sigma_copper = 6e7
sigma = mesh.MaterialCF({"slot(.*)_bundle.*" : sigma_copper})

# Display conductivity distribution
print("Conductivity (S/m)")
Draw(sigma, mesh,
     settings = {"Objects" : {"Wireframe" : False}},
     filename = "scenes/ref/sigma.html")

In [ ]:
# Rotating remanence / magnetization

Br_magnets = 1 # remanence in T

# Define magnetization in A/m necessary for the magnetic vector-potential formulation
from utils.physics import magnetization_halbach
M = mesh.MaterialCF({"rotor" : magnetization_halbach(br = Br_magnets, p = pole_pairs)}) # A/m

# Display remanence distribution
electric_angle = 0 # angle can be changed, try 90 or 180°
print(f"Remanent flux density (T) at an electric angle of {electric_angle:.0f}°")
from ngsolve import CF, exp
Draw(mu0 * CF(((M[0]*exp(1j*pi/180*electric_angle)).real, 
               (M[1]*exp(1j*pi/180*electric_angle)).real)), mesh,
     vectors = {"grid_size": 40},
     settings = {"Objects" : {"Wireframe" : False, "Surface": False}},
     filename = f"scenes/ref/Br_{electric_angle:.0f}°el.html")

### 2.b) Current supply

In [ ]:
# Define the supply parmeters

I_rms = 1            # A
load_angle = 0       # rad
frequency = 1000     # Hz

# Distribute the current through conductors
from utils.supply import phase_current
phase = phase_current(I_rms=I_rms, load_angle=load_angle)

# Display the A-B-C currents
import matplotlib.pyplot as plt
import numpy as np
from ngsolve import pi, exp
theta = np.linspace(0, 4*pi, 100)
plt.plot(theta, (phase["Ap"] * exp(1j * theta)).real, label = "A")
plt.plot(theta, (phase["Bp"] * exp(1j * theta)).real, label = "B")
plt.plot(theta, (phase["Cp"] * exp(1j * theta)).real, label = "C")
plt.xlabel("Electrical angle (rad)"); plt.ylabel("I (A)"); 
plt.grid(); plt.legend(); plt.title("3-phase current supply")
plt.show()

In [ ]:
# Define the winding arangement
winding_type = "distributed" # "concentrated"

# Distribute the current within the conductors
from utils.supply import winding_arrangement, bundle_arrangement
winding = winding_arrangement(phase = phase, type = winding_type)
bundles = bundle_arrangement(winding = winding, bundles_per_half_slot = bundles_per_half_slot)

# Display current distribution 
electric_angle = 0 # angle can be changed, try 90 or 180
print(f"Current (A) at an electric angle of {electric_angle:.0f}°")
Draw((mesh.MaterialCF(bundles)*exp(1j*pi/180*electric_angle)).real, mesh,
     settings = {"Objects" : {"Wireframe" : False}},
     filename = f"scenes/ref/J_{electric_angle:.0f}°el.html")